# 055 — Atención y arquitectura Transformer

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Atención escalada** (Vaswani et al., 2017): cada token emite query, key y value;
`Attention(Q,K,V) = softmax(Q·Kᵀ/√d_k)·V`. El escalado √d_k evita saturar el softmax;
la salida de cada token es una media ponderada de los values de todos los tokens:
cualquier par de posiciones se conecta a distancia 1.

**Multi-head**: h atenciones paralelas de dimensión d_model/h, concatenadas.
**Máscara causal**: −∞ sobre posiciones futuras → entrenamiento paralelo de un
generador autoregresivo. **Codificación posicional** (senos/cosenos o aprendida):
imprescindible, la atención es invariante al orden.

**Bloque Transformer**: atención (mezcla entre tokens) + FFN por token, ambos con
residual + layer norm. Coste O(n²·d) en la longitud de secuencia.


## 🧮 Ejemplo de referencia (atención 2×2)

Q=K=I, V=[[1,2],[3,4]], d_k=2 → puntuaciones escaladas diag(0.7071);
softmax fila → [[0.6698, 0.3302], [0.3302, 0.6698]];
salida = [[1.6604, 2.6604], [2.3396, 3.3396]].

Con máscara causal, el token 1 solo se ve a sí mismo: su salida sería (1,2) exacta.
Reproducir estas cuentas a mano es el ejercicio central de esta clase.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("attention", seed=55)
show(result)


## Reflexión

1. ¿Qué problema concreto de la LSTM resuelve que el camino entre dos posiciones cualesquiera sea O(1), y qué precio computacional se paga (O(n²))?
2. Si eliminas la codificación posicional de un Transformer entrenado, ¿qué tareas seguirían funcionando y cuáles colapsarían?
3. ¿Por qué la máscara causal permite entrenar en paralelo un modelo que en inferencia es estrictamente secuencial?
